# 03 - Gold Curated (Modelagem Analítica)

Nesta etapa, construímos nosso modelo dimensional (**Star Schema**) contendo Dimensões e Fatos, otimizado para que ferramentas de BI e analistas de negócio possam extrair indicadores rapidamente (Receita, Ticket Médio, Atrasos, SAC).

**Decisão Técnica**: Utilizamos **Spark SQL** para as transformações da camada Gold, garantindo máxima clareza nas regras de negócio e joins.

In [0]:
%run ./utils

# Utils

Notebook contendo funções reutilizáveis para o pipeline de dados.

In [0]:
import pyspark.sql.functions as F

# Registrar todas as tabelas Silver como Views Temporárias para uso no SQL
tables = ["clientes", "produtos", "vendedores", "regioes", "canais", "pedidos", "pedidos_itens", "entregas", "ocorrencias"]

for table in tables:
    spark.read.table(f"{catalog_name}.silver.{table}").createOrReplaceTempView(f"silver_{table}")

print("Views temporárias da camada Silver criadas com sucesso.")

Views temporárias da camada Silver criadas com sucesso.


## 1. Tabelas de Dimensão

In [0]:
print("Criando Dimensões...")

# 1.1 Dimensão Clientes
dim_clientes = spark.sql("""
    SELECT * EXCEPT (_silver_processed_at), current_timestamp() as _gold_processed_at 
    FROM silver_clientes
""")
save_delta_table(dim_clientes, "dim_clientes", "gold", catalog=catalog_name)

# 1.2 Dimensão Produtos
dim_produtos = spark.sql("""
    SELECT * EXCEPT (_silver_processed_at), current_timestamp() as _gold_processed_at 
    FROM silver_produtos
""")
save_delta_table(dim_produtos, "dim_produtos", "gold", catalog=catalog_name)

# 1.3 Dimensão Vendedores (CONSOLIDADA: Vendedores + Canais + Regiões)
dim_vendedores = spark.sql("""
    SELECT 
        v.seller_id,
        v.seller_name,
        v.hire_date,
        v.flg_ativo as seller_ativo,
        c.nome_canal,
        c.tipo_canal,
        r.regional_name,
        r.state as regional_state,
        r.manager_name,
        current_timestamp() as _gold_processed_at

    FROM silver_vendedores v

    LEFT JOIN silver_canais c ON v.canal_id = c.id_canal
    LEFT JOIN silver_regioes r ON v.regional_code = r.regional_code
""")

save_delta_table(dim_vendedores, "dim_vendedores", "gold", catalog=catalog_name)

print("Dimensões Gold criadas com sucesso!")

Criando Dimensões...
Salvando tabela gerenciada: workspace.gold.dim_clientes no modo overwrite...
Salvo com sucesso!
Salvando tabela gerenciada: workspace.gold.dim_produtos no modo overwrite...
Salvo com sucesso!
Salvando tabela gerenciada: workspace.gold.dim_vendedores no modo overwrite...
Salvo com sucesso!
Dimensões Gold criadas com sucesso!


## 2. Tabelas de Fato

In [0]:
print("Criando Fatos...")

# 2.1 Fato Vendas (Granularidade: Item de Pedido)
fact_vendas = spark.sql("""
    SELECT 
        i.pedido_id,
        i.item_id,
        i.product_id,
        p.customer_id,
        p.vendedor_id,
        p.data_pedido,
        p.status_pedido,
        p.prioridade,
        p.origem_pedido,
        i.quantidade,
        i.preco_unitario,
        i.valor_item as valor_bruto_item,
        CAST((i.valor_item / NULLIF(p.valor_bruto, 0)) * p.valor_desconto AS DOUBLE) as valor_desconto_item,
        CAST(i.valor_item - ((i.valor_item / NULLIF(p.valor_bruto, 0)) * p.valor_desconto) AS DOUBLE) as valor_liquido_item,
        (p.flg_qualidade AND i.flg_qualidade) as flg_qualidade_venda,
        current_timestamp() as _gold_processed_at

    FROM silver_pedidos_itens i

    INNER JOIN silver_pedidos p ON i.pedido_id = p.pedido_id
""")
save_delta_table(fact_vendas, "fact_vendas", "gold", catalog=catalog_name)
optimize_table("fact_vendas", "gold", zorder_cols=["pedido_id", "product_id", "data_pedido"], catalog=catalog_name)

# 2.2 Fato Entregas e SLA
fact_entregas = spark.sql("""
    SELECT 
        e.delivery_id,
        e.order_id as pedido_id,
        p.data_pedido,
        p.data_previsao_entrega,
        e.shipped_at as data_envio,
        e.delivered_at as data_entrega,
        e.delivery_status,
        e.carrier_name,
        e.cost as custo_frete,
        DATEDIFF(e.delivered_at, p.data_previsao_entrega) as dias_atraso,
        CASE WHEN e.delivered_at > p.data_previsao_entrega THEN 1 ELSE 0 END as flg_atraso,
        e.flg_qualidade as flg_qualidade_entrega,
        current_timestamp() as _gold_processed_at

    FROM silver_entregas e

    INNER JOIN silver_pedidos p ON e.order_id = p.pedido_id
""")
save_delta_table(fact_entregas, "fact_entregas", "gold", catalog=catalog_name)
optimize_table("fact_entregas", "gold", zorder_cols=["pedido_id", "delivery_id"], catalog=catalog_name)

print("Fatos Gold criadas com sucesso!")

Criando Fatos...
Salvando tabela gerenciada: workspace.gold.fact_vendas no modo overwrite...
Salvo com sucesso!
Executando otimização: OPTIMIZE workspace.gold.fact_vendas ZORDER BY (pedido_id, product_id, data_pedido)...
Otimização finalizada com sucesso!
Salvando tabela gerenciada: workspace.gold.fact_entregas no modo overwrite...
Salvo com sucesso!
Executando otimização: OPTIMIZE workspace.gold.fact_entregas ZORDER BY (pedido_id, delivery_id)...
Otimização finalizada com sucesso!
Fatos Gold criadas com sucesso!


## 3. Exemplos de Consultas Analíticas (BI-Ready)

In [0]:
# KPI: Receita Líquida e Ticket Médio por Região
spark.sql(f"""
    SELECT 
        v.regional_name,
        SUM(f.valor_liquido_item) as receita_liquida,
        COUNT(DISTINCT f.pedido_id) as qtd_pedidos,
        SUM(f.valor_liquido_item) / COUNT(DISTINCT f.pedido_id) as ticket_medio
    FROM {catalog_name}.gold.fact_vendas f
    JOIN {catalog_name}.gold.dim_vendedores v ON f.vendedor_id = v.seller_id
    WHERE f.flg_qualidade_venda = true
    GROUP BY 1
    ORDER BY receita_liquida DESC
""").show()